# 🔬 Deep Research Agent 實作

**LLM Application - AI Agent 課程｜Day 2 下午**

在這個 Notebook 中，我們將實作一個完整的 Deep Research Agent，能夠：
- 自動拆解研究問題為子問題
- 搜尋並閱讀相關資料
- 彙整並自我評審研究結果
- 迭代式地深化研究直到品質達標

---

In [ ]:
%%capture
!pip install langchain langgraph langchain-openai requests beautifulsoup4

In [ ]:
import os

# 請填入你的 OpenAI API Key
os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

---

# Part 1: Deep Research 概念與業界實踐

---

## 1.1 什麼是 Deep Research？

**Deep Research**（深度研究）是一種 AI Agent 能力，讓模型能夠像研究員一樣：

1. **主動探索**：不只回答問題，而是主動搜尋、閱讀、分析多個資料來源
2. **迭代深化**：根據初步發現，產生新的問題並繼續研究
3. **批判性思考**：自我評估研究品質，發現資訊缺口

### Deep Research vs 簡單 Q&A vs RAG

| 特性 | 簡單 Q&A | RAG | Deep Research Agent |
|------|---------|-----|-------------------|
| 資料來源 | 模型訓練資料 | 預建向量資料庫 | 即時搜尋 + 網頁閱讀 |
| 研究深度 | 單次回答 | 單次檢索 + 回答 | 多輪迭代研究 |
| 主動性 | 被動回答 | 被動檢索 | 主動探索 |
| 品質控制 | 無 | 無 | 自我評審 + 迭代改進 |
| 適用場景 | 簡單事實查詢 | 企業知識庫問答 | 複雜研究報告 |

### 業界實踐

- **OpenAI Deep Research**：GPT-4 結合瀏覽器工具，可花數分鐘進行深度網路研究
- **Perplexity**：即時搜尋 + 多來源整合 + 引用標注
- **Google Gemini Deep Research**：多輪搜尋 + 自動產生研究計畫
- **Tavily**：專為 AI Agent 設計的搜尋 API

## 1.2 研究型 Agent 流程拆解

Deep Research Agent 的核心是一個**迭代式研究迴圈**：

```
                    ┌─────────────────────────────────────┐
                    │                                     │
                    ▼                                     │
┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────────────┐
│  Planner │──▶│ Searcher │──▶│  Reader  │──▶│ Synthesizer +    │
│ 研究規劃  │   │ 資料搜尋  │   │ 內容閱讀  │   │ Critic 彙整評審   │
└──────────┘   └──────────┘   └──────────┘   └──────────────────┘
                                                    │
                                              ┌─────┴─────┐
                                              │           │
                                         品質不足      品質足夠
                                         (迴圈)       (結束)
                                              │           │
                                              │     ┌─────▼──────┐
                                              │     │ Final Report│
                                              │     │  最終報告    │
                                              │     └────────────┘
                                              │
                                    ┌─────────▼─────────┐
                                    │ 產生新子問題        │
                                    │ 補充不足的資訊      │
                                    └───────────────────┘
```

每一輪迭代都在加深研究的深度和廣度，直到品質達標或達到最大迭代次數。

## 1.3 Research Agent vs Agentic RAG 的差異

| 比較面向 | Agentic RAG | Deep Research Agent |
|---------|-------------|-------------------|
| **核心行為** | 被動檢索：從既有知識庫找答案 | 主動探索：搜尋、閱讀、發現新資訊 |
| **資料範圍** | 預先建立的向量資料庫 | 整個網路 / 即時搜尋結果 |
| **研究策略** | 查詢改寫 → 檢索 → 回答 | 規劃 → 搜尋 → 閱讀 → 彙整 → 評審 → 迭代 |
| **迭代能力** | 可能有查詢改寫的迴圈 | 完整的研究迭代，含自我評審 |
| **品質控制** | 通常只看相關性分數 | 自我評審：完整性、證據充分性、矛盾檢測 |
| **輸出格式** | 簡短回答 + 引用 | 結構化研究報告 |
| **適用場景** | 企業內部知識問答 | 開放性研究問題、市場調查、技術分析 |

**關鍵差異：**
- RAG 是「找到答案」—— 從已有資料中檢索
- Research Agent 是「研究出答案」—— 主動探索、發現、綜合新知識

---

# Part 2: Deep Research Agent 實作

---

## 2.1 系統架構規劃

```
┌─────────────────────────────────────────────────────────────┐
│                  Deep Research Agent                         │
│                                                             │
│  ┌──────────┐                                               │
│  │  START   │                                               │
│  └────┬─────┘                                               │
│       ▼                                                     │
│  ┌──────────┐    研究問題 → 3~5 個子問題                      │
│  │ Planner  │                                               │
│  └────┬─────┘                                               │
│       ▼                                                     │
│  ┌──────────┐    每個子問題 → 搜尋結果                        │
│  │ Searcher │                                               │
│  └────┬─────┘                                               │
│       ▼                                                     │
│  ┌──────────┐    搜尋結果 → 摘要內容                          │
│  │  Reader  │                                               │
│  └────┬─────┘                                               │
│       ▼                                                     │
│  ┌────────────────┐  彙整 + 自我評審                         │
│  │ Synth + Critic │──────┐                                  │
│  └────────────────┘      │                                  │
│       │            品質不足 → 回到 Planner                    │
│       │ 品質足夠                                             │
│       ▼                                                     │
│  ┌──────────────┐                                           │
│  │ Final Report │                                           │
│  └──────┬───────┘                                           │
│         ▼                                                   │
│     ┌──────┐                                                │
│     │ END  │                                                │
│     └──────┘                                                │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
from typing import Optional
from typing_extensions import TypedDict

# ===== 定義研究狀態 =====
class ResearchState(TypedDict):
    research_question: str          # 原始研究問題
    sub_questions: list             # 拆解後的子問題
    current_sub_question: Optional[str]  # 當前處理的子問題
    search_results: list            # 累積的搜尋結果
    page_contents: list             # 擷取的網頁內容
    sub_answers: list               # 各子問題的答案
    synthesis: Optional[str]        # 彙整後的報告
    critique: Optional[str]         # 自我評審意見
    is_sufficient: bool             # 研究是否足夠完整
    iteration: int                  # 當前迭代次數
    max_iterations: int             # 最大迭代次數
    final_report: Optional[str]     # 最終輸出報告

print("ResearchState 定義完成！")
print("狀態包含：研究問題、子問題、搜尋結果、內容摘要、彙整報告、評審意見等")

In [ ]:
from langchain_openai import ChatOpenAI

# 初始化 LLM
llm = ChatOpenAI(model="gpt-4.1", temperature=0.3)

print("LLM 初始化完成：gpt-4.1")

## 2.2 Planner 研究規劃節點

Planner 的職責：
1. 接收研究問題
2. 用 LLM 將問題拆解成 3~5 個可搜尋的子問題
3. 如果是第二輪以上的迭代，根據 critique 產生補充子問題

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
import json

def planner_node(state: ResearchState) -> dict:
    """研究規劃節點：將研究問題拆解為子問題"""

    iteration = state.get("iteration", 0)
    research_question = state["research_question"]

    # 第一輪：從零開始拆解問題
    if iteration == 0:
        system_prompt = """你是一位研究規劃專家。你的任務是將一個研究問題拆解成 3~5 個具體、可搜尋的子問題。

要求：
1. 每個子問題都應該是具體且可搜尋的
2. 子問題要涵蓋研究問題的不同面向
3. 子問題之間不要有太多重疊

請以 JSON 格式回傳，格式如下：
{"sub_questions": ["子問題1", "子問題2", "子問題3"]}

只回傳 JSON，不要其他文字。"""
        user_msg = f"請將以下研究問題拆解為子問題：\n\n{research_question}"

    # 後續輪次：根據 critique 補充新的子問題
    else:
        system_prompt = """你是一位研究規劃專家。根據之前的研究評審意見，你需要產生 2~3 個新的補充子問題來填補資訊缺口。

要求：
1. 新子問題要針對評審中指出的不足之處
2. 不要重複之前已經研究過的問題
3. 子問題要具體且可搜尋

請以 JSON 格式回傳，格式如下：
{"sub_questions": ["補充子問題1", "補充子問題2"]}

只回傳 JSON，不要其他文字。"""
        critique = state.get("critique", "無評審意見")
        existing_questions = state.get("sub_questions", [])
        user_msg = f"""原始研究問題：{research_question}

之前已研究的子問題：{json.dumps(existing_questions, ensure_ascii=False)}

評審意見：{critique}

請產生補充子問題："""

    # 呼叫 LLM
    messages = [SystemMessage(content=system_prompt), HumanMessage(content=user_msg)]
    response = llm.invoke(messages)

    # 解析回傳的 JSON
    try:
        result = json.loads(response.content)
        new_questions = result["sub_questions"]
    except (json.JSONDecodeError, KeyError):
        # 如果 JSON 解析失敗，嘗試用簡單方式提取
        new_questions = [response.content]

    # 累積子問題（後續輪次追加新問題）
    all_questions = state.get("sub_questions", []) + new_questions

    print(f"\n📋 [Planner] 第 {iteration + 1} 輪規劃")
    print(f"研究問題：{research_question}")
    print(f"產生的子問題：")
    for i, q in enumerate(new_questions, 1):
        print(f"  {i}. {q}")

    return {
        "sub_questions": all_questions,
        "iteration": iteration + 1,
    }

# 測試 Planner
test_state = {
    "research_question": "2024年台灣AI產業的發展趨勢與主要挑戰",
    "sub_questions": [],
    "iteration": 0,
    "max_iterations": 3,
}

result = planner_node(test_state)
print(f"\n子問題總數：{len(result['sub_questions'])}")

## 2.3 Searcher 資料搜尋節點

Searcher 的職責：
1. 對每個子問題執行搜尋
2. 收集搜尋結果（標題、URL、摘要）
3. 在課堂上，我們使用**模擬搜尋函式**，讓同學不需要搜尋引擎 API Key 也能執行

> **提示**：實際應用時可替換為 Tavily API、Google Search API、Bing API 等

In [ ]:
import random

# ===== 模擬搜尋函式 =====
# 課堂用：回傳看起來很真實的搜尋結果，不需要真實 API Key

MOCK_SEARCH_DB = {
    "台灣AI產業": [
        {"title": "2024台灣AI產業全景報告 - 資策會MIC", "url": "https://mic.iii.org.tw/ai-report-2024", "snippet": "2024年台灣AI產業產值突破新台幣5,000億元，年增率達35%。生成式AI應用帶動企業數位轉型加速，半導體產業持續在AI晶片領域占據全球領先地位。"},
        {"title": "台灣AI發展現況與挑戰 - 工研院", "url": "https://www.itri.org.tw/ai-taiwan-2024", "snippet": "台灣在AI硬體（GPU、AI晶片）全球市佔率超過60%，但在AI軟體應用和人才培育方面仍有顯著落差。2024年AI新創公司數量成長40%。"},
        {"title": "經濟部AI產業推動策略白皮書", "url": "https://www.moea.gov.tw/ai-strategy-2024", "snippet": "政府計畫在2025年前投入500億元推動AI產業，重點包括：AI人才培育、產業AI化、AI產業化三大方向。"},
    ],
    "AI發展趨勢": [
        {"title": "2024 全球AI十大趨勢 - Gartner", "url": "https://www.gartner.com/ai-trends-2024", "snippet": "2024年AI關鍵趨勢：1)多模態AI普及 2)AI Agent自主化 3)小型語言模型崛起 4)AI治理法規成形 5)企業AI ROI開始顯現。"},
        {"title": "生成式AI商業應用現況 - McKinsey", "url": "https://www.mckinsey.com/genai-business-2024", "snippet": "調查顯示65%的企業已在至少一個業務流程中採用生成式AI，最常見的應用場景為客服自動化(35%)、內容生成(28%)、程式碼輔助(22%)。"},
        {"title": "AI Agent 技術發展路線圖 - Stanford HAI", "url": "https://hai.stanford.edu/ai-agent-roadmap", "snippet": "AI Agent 被認為是下一波AI應用的核心，2024年各大科技公司紛紛推出Agent框架，從工具呼叫到自主規劃能力持續進化。"},
    ],
    "AI人才": [
        {"title": "台灣AI人才缺口分析 - 104人力銀行", "url": "https://www.104.com.tw/ai-talent-2024", "snippet": "2024年台灣AI相關職缺數量達4.2萬個，年增55%。AI工程師平均月薪8-15萬，但人才缺口仍達1.5萬人。企業反映最缺乏具實務經驗的AI應用人才。"},
        {"title": "台灣AI教育現況與展望", "url": "https://www.edu.tw/ai-education-2024", "snippet": "全台已有超過50所大學設立AI相關系所或學程，但業界反映學用落差仍大。教育部推動AI融入各科教學計畫。"},
    ],
    "AI挑戰": [
        {"title": "台灣推動AI面臨的五大挑戰 - 天下雜誌", "url": "https://www.cw.com.tw/ai-challenges-taiwan", "snippet": "主要挑戰包括：1)資料取得與隱私法規限制 2)AI人才外流至國際大廠 3)中小企業AI導入門檻高 4)繁體中文AI模型發展落後 5)AI倫理與治理框架不完善。"},
        {"title": "企業AI轉型的困境與突破 - 數位時代", "url": "https://www.bnext.com.tw/ai-transformation-2024", "snippet": "調查指出70%台灣企業認為AI重要，但僅25%已有具體AI策略。主要障礙為缺乏AI人才(45%)、資料品質不佳(30%)、投資回報不確定(25%)。"},
    ],
    "AI晶片": [
        {"title": "台積電AI晶片佈局分析 - 財經M平方", "url": "https://www.macromicro.me/tsmc-ai-chip-2024", "snippet": "台積電在先進製程（3nm、2nm）持續領先，AI晶片代工營收佔比從2023年的15%提升至2024年的25%。N3E製程良率已突破80%。"},
        {"title": "NVIDIA與台灣AI生態系 - 科技新報", "url": "https://technews.tw/nvidia-taiwan-2024", "snippet": "NVIDIA在台灣建立完整的AI生態系，與鴻海、廣達、華碩等合作開發AI伺服器。台灣AI伺服器全球市佔率超過90%。"},
    ],
    "AI法規": [
        {"title": "台灣AI基本法立法進度 - 立法院", "url": "https://www.ly.gov.tw/ai-law-2024", "snippet": "台灣《人工智慧基本法》草案已進入立法院審議階段，預計2025年通過。內容涵蓋AI使用原則、風險分級管理、資料治理等。"},
        {"title": "全球AI監管趨勢對台灣的啟示", "url": "https://www.tier.org.tw/ai-regulation-global", "snippet": "歐盟AI Act已正式生效，美國、日本也陸續推出AI治理框架。台灣需要在促進創新與管控風險之間取得平衡。"},
    ],
}

def mock_web_search(query: str, num_results: int = 3) -> list:
    """模擬網路搜尋，根據關鍵字比對回傳搜尋結果"""
    results = []

    # 根據查詢關鍵字比對模擬資料
    for keyword, items in MOCK_SEARCH_DB.items():
        if any(k in query for k in keyword):
            results.extend(items)

    # 如果沒有比對到，回傳通用結果
    if not results:
        results = [
            {"title": f"關於「{query}」的研究報告", "url": f"https://example.com/search?q={query}", "snippet": f"這是一份關於「{query}」的綜合研究報告，涵蓋最新發展趨勢、市場分析與未來展望。"},
            {"title": f"「{query}」深度分析 - 產業研究", "url": f"https://example.com/analysis/{query}", "snippet": f"深入分析「{query}」的各個面向，包含技術發展、市場規模、競爭格局與政策影響。"},
        ]

    # 隨機打亂並限制數量
    random.shuffle(results)
    return results[:num_results]

# 測試模擬搜尋
test_results = mock_web_search("台灣AI產業發展趨勢")
print("🔍 搜尋測試結果：")
for r in test_results:
    print(f"  - {r['title']}")
    print(f"    {r['snippet'][:60]}...")
    print()

In [ ]:
def searcher_node(state: ResearchState) -> dict:
    """搜尋節點：對每個子問題進行搜尋"""

    sub_questions = state["sub_questions"]
    existing_results = state.get("search_results", [])

    # 只搜尋還沒搜尋過的子問題（已搜尋的子問題數 = 已有結果的組數）
    already_searched = len(existing_results)
    new_questions = sub_questions[already_searched:]

    new_results = []
    for question in new_questions:
        print(f"🔍 [Searcher] 搜尋：{question}")
        results = mock_web_search(question, num_results=3)
        new_results.append({
            "sub_question": question,
            "results": results,
        })
        for r in results:
            print(f"    → {r['title']}")

    # 累積搜尋結果
    all_results = existing_results + new_results

    print(f"\n📊 搜尋完成，共 {len(all_results)} 組結果")
    return {"search_results": all_results}

# 測試 Searcher
test_state_2 = {
    **test_state,
    "sub_questions": result["sub_questions"],
    "search_results": [],
}

search_result = searcher_node(test_state_2)

## 2.4 Reader 內容閱讀節點

Reader 的職責：
1. 從搜尋結果中「閱讀」網頁內容（這裡使用模擬資料 + LLM 摘要）
2. 針對每個子問題，整理出相關資訊摘要
3. 實際應用中，可用 `requests + BeautifulSoup` 抓取網頁內容

In [ ]:
import requests
from bs4 import BeautifulSoup

def fetch_page_content(url: str) -> str:
    """嘗試擷取網頁內容（實際環境使用）"""
    try:
        response = requests.get(url, timeout=10, headers={
            "User-Agent": "Mozilla/5.0 (Research Agent)"
        })
        soup = BeautifulSoup(response.text, "html.parser")

        # 移除 script 和 style 標籤
        for tag in soup(["script", "style", "nav", "footer"]):
            tag.decompose()

        # 取得純文字
        text = soup.get_text(separator="\n", strip=True)
        return text[:3000]  # 限制長度
    except Exception as e:
        return f"無法擷取網頁內容：{e}"

def reader_node(state: ResearchState) -> dict:
    """閱讀節點：閱讀搜尋結果並產生摘要"""

    search_results = state["search_results"]
    existing_contents = state.get("page_contents", [])
    existing_answers = state.get("sub_answers", [])

    # 只處理新的搜尋結果
    already_read = len(existing_contents)
    new_search_groups = search_results[already_read:]

    new_contents = []
    new_answers = []

    for group in new_search_groups:
        sub_question = group["sub_question"]
        results = group["results"]

        print(f"\n📖 [Reader] 閱讀關於：{sub_question}")

        # 收集所有搜尋結果的摘要作為「閱讀內容」
        # （課堂環境用 snippet 模擬，實際可用 fetch_page_content）
        combined_content = ""
        sources = []
        for r in results:
            combined_content += f"來源：{r['title']}\n內容：{r['snippet']}\n\n"
            sources.append({"title": r["title"], "url": r["url"]})

        new_contents.append({
            "sub_question": sub_question,
            "raw_content": combined_content,
            "sources": sources,
        })

        # 用 LLM 針對子問題做摘要回答
        messages = [
            SystemMessage(content="你是一位研究助理。根據提供的資料，針對問題給出簡潔但完整的回答。請引用具體數據和事實。用繁體中文回答。"),
            HumanMessage(content=f"問題：{sub_question}\n\n參考資料：\n{combined_content}\n\n請根據以上資料回答問題：")
        ]
        response = llm.invoke(messages)

        new_answers.append({
            "sub_question": sub_question,
            "answer": response.content,
            "sources": sources,
        })

        print(f"  ✅ 摘要完成（{len(response.content)} 字）")

    # 累積結果
    all_contents = existing_contents + new_contents
    all_answers = existing_answers + new_answers

    return {
        "page_contents": all_contents,
        "sub_answers": all_answers,
    }

print("Reader 節點定義完成！")

## 2.5 Synthesizer + Critic 彙整評審節點

這是 Deep Research Agent 最關鍵的節點，負責：
1. **Synthesizer（彙整者）**：將所有子問題的答案整合成連貫的研究報告
2. **Critic（評審者）**：自我評估報告品質
   - 所有子問題是否都有被回答？
   - 證據是否充分？
   - 是否有矛盾或資訊缺口？

In [ ]:
def synthesizer_critic_node(state: ResearchState) -> dict:
    """彙整 + 評審節點：整合答案並自我評估品質"""

    research_question = state["research_question"]
    sub_answers = state["sub_answers"]
    iteration = state["iteration"]

    # ===== 第一步：彙整所有子答案 =====
    answers_text = ""
    for sa in sub_answers:
        answers_text += f"\n### 子問題：{sa['sub_question']}\n"
        answers_text += f"{sa['answer']}\n"
        answers_text += f"來源：{', '.join([s['title'] for s in sa['sources']])}\n"

    synth_messages = [
        SystemMessage(content="""你是一位資深研究分析師。請將以下各子問題的研究發現整合成一份連貫的研究摘要。

要求：
1. 用繁體中文撰寫
2. 整合不同子問題的發現，找出共同主題和關聯
3. 引用具體數據
4. 結構清晰，用標題分段
5. 控制在 500-800 字"""),
        HumanMessage(content=f"研究問題：{research_question}\n\n各子問題研究結果：\n{answers_text}\n\n請整合以上內容成為研究摘要：")
    ]

    synth_response = llm.invoke(synth_messages)
    synthesis = synth_response.content

    print(f"\n📝 [Synthesizer] 彙整完成（{len(synthesis)} 字）")

    # ===== 第二步：自我評審 =====
    critic_messages = [
        SystemMessage(content="""你是一位嚴格的研究品質評審員。請評估以下研究摘要的品質。

請以 JSON 格式回覆，包含：
{
    "is_sufficient": true/false,
    "critique": "評審意見文字",
    "missing_aspects": ["缺少的面向1", "缺少的面向2"],
    "score": 1-10
}

評分標準：
- 8分以上：研究充分，可以產出最終報告
- 5-7分：有一些缺口，建議補充研究
- 5分以下：嚴重不足，必須補充

只回傳 JSON，不要其他文字。"""),
        HumanMessage(content=f"""原始研究問題：{research_question}

研究摘要：
{synthesis}

目前是第 {iteration} 輪研究，最多 {state['max_iterations']} 輪。

請評審此研究摘要的品質：""")
    ]

    critic_response = llm.invoke(critic_messages)

    # 解析評審結果
    try:
        critic_result = json.loads(critic_response.content)
        is_sufficient = critic_result.get("is_sufficient", False)
        critique = critic_result.get("critique", "無評審意見")
        score = critic_result.get("score", 5)
    except (json.JSONDecodeError, KeyError):
        is_sufficient = False
        critique = critic_response.content
        score = 5

    print(f"🔍 [Critic] 評審分數：{score}/10")
    print(f"   是否充分：{'是' if is_sufficient else '否'}")
    print(f"   評審意見：{critique[:100]}...")

    return {
        "synthesis": synthesis,
        "critique": critique,
        "is_sufficient": is_sufficient,
    }

print("Synthesizer + Critic 節點定義完成！")

## 2.6 Conditional Edge：自適應研究終止邏輯

這是控制「繼續研究」或「結束研究」的條件邊（Conditional Edge）：

- 如果 `is_sufficient == True` **或** `iteration >= max_iterations` → 進入 Final Report
- 如果 `is_sufficient == False` → 回到 Planner 產生新的補充子問題，繼續研究迴圈

In [ ]:
def should_continue_research(state: ResearchState) -> str:
    """條件判斷：決定是繼續研究還是產出最終報告"""

    is_sufficient = state.get("is_sufficient", False)
    iteration = state.get("iteration", 0)
    max_iterations = state.get("max_iterations", 3)

    if is_sufficient:
        print(f"\n✅ 研究品質足夠，進入最終報告階段")
        return "final_report"

    if iteration >= max_iterations:
        print(f"\n⏰ 已達最大迭代次數（{max_iterations}），進入最終報告階段")
        return "final_report"

    print(f"\n🔄 研究不足，進入第 {iteration + 1} 輪研究（最多 {max_iterations} 輪）")
    return "continue_research"

# 測試條件邏輯
print("測試 1 - 品質足夠：", should_continue_research({"is_sufficient": True, "iteration": 1, "max_iterations": 3}))
print("測試 2 - 達到上限：", should_continue_research({"is_sufficient": False, "iteration": 3, "max_iterations": 3}))
print("測試 3 - 需要繼續：", should_continue_research({"is_sufficient": False, "iteration": 1, "max_iterations": 3}))

## 2.7 Final Report 最終報告節點

最終報告節點將所有研究成果整合成一份結構化的研究報告，包含：
- 摘要
- 各面向的研究發現
- 資料來源
- 研究限制

In [ ]:
def final_report_node(state: ResearchState) -> dict:
    """最終報告節點：產生結構化的研究報告"""

    research_question = state["research_question"]
    sub_answers = state["sub_answers"]
    synthesis = state.get("synthesis", "")
    iteration = state["iteration"]

    # 收集所有來源
    all_sources = []
    for sa in sub_answers:
        for src in sa.get("sources", []):
            if src not in all_sources:
                all_sources.append(src)

    # 組合子答案資訊
    answers_text = ""
    for sa in sub_answers:
        answers_text += f"\n子問題：{sa['sub_question']}\n回答：{sa['answer']}\n"

    # 用 LLM 產生最終報告
    messages = [
        SystemMessage(content="""你是一位專業的研究報告撰寫者。請根據所有研究發現，撰寫一份完整的研究報告。

報告格式要求：
1. **研究摘要**（Executive Summary）：200字以內的研究重點摘要
2. **研究發現**：按主題分段呈現詳細發現，引用具體數據
3. **關鍵洞察**：跨主題的重要發現和趨勢
4. **結論與建議**：總結和未來展望
5. **資料來源**：列出所有參考來源
6. **研究限制**：說明研究的侷限性

用繁體中文撰寫，語氣專業但易讀。"""),
        HumanMessage(content=f"""研究問題：{research_question}

研究彙整：
{synthesis}

各子問題詳細回答：
{answers_text}

共進行了 {iteration} 輪研究迭代。

請撰寫完整研究報告：""")
    ]

    response = llm.invoke(messages)

    # 附加來源清單
    sources_text = "\n\n---\n### 📚 參考來源\n"
    for i, src in enumerate(all_sources, 1):
        sources_text += f"{i}. [{src['title']}]({src['url']})\n"

    final_report = response.content + sources_text

    print(f"\n📄 [Final Report] 最終報告產生完成！")
    print(f"   報告長度：{len(final_report)} 字")
    print(f"   參考來源：{len(all_sources)} 個")
    print(f"   研究迭代：{iteration} 輪")

    return {"final_report": final_report}

print("Final Report 節點定義完成！")

## 2.8 組裝完整 Deep Research Agent

現在將所有節點用 LangGraph 串接起來：

```
START → Planner → Searcher → Reader → Synthesizer_Critic
    ↗                                        ↓
Planner ← (不足) ←── Conditional Edge ──→ (足夠) → Final_Report → END
```

In [ ]:
from langgraph.graph import StateGraph, START, END

# ===== 建立 Graph =====
workflow = StateGraph(ResearchState)

# 加入所有節點
workflow.add_node("planner", planner_node)
workflow.add_node("searcher", searcher_node)
workflow.add_node("reader", reader_node)
workflow.add_node("synthesizer_critic", synthesizer_critic_node)
workflow.add_node("final_report", final_report_node)

# 設定邊：線性流程
workflow.add_edge(START, "planner")
workflow.add_edge("planner", "searcher")
workflow.add_edge("searcher", "reader")
workflow.add_edge("reader", "synthesizer_critic")

# 條件邊：根據評審結果決定繼續或結束
workflow.add_conditional_edges(
    "synthesizer_critic",
    should_continue_research,
    {
        "continue_research": "planner",      # 不足 → 回到 Planner
        "final_report": "final_report",       # 足夠 → 最終報告
    }
)

# 最終報告 → 結束
workflow.add_edge("final_report", END)

# 編譯 Graph
graph = workflow.compile()

print("Deep Research Agent 組裝完成！")

In [ ]:
# 視覺化 Graph
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    # 如果無法產生圖片，印出 Mermaid 文字
    print("Graph 結構（Mermaid 格式）：")
    print(graph.get_graph().draw_mermaid())

In [ ]:
# ===== 執行 Deep Research Agent =====

# 初始狀態
initial_state = {
    "research_question": "2024年台灣AI產業的發展趨勢與主要挑戰",
    "sub_questions": [],
    "current_sub_question": None,
    "search_results": [],
    "page_contents": [],
    "sub_answers": [],
    "synthesis": None,
    "critique": None,
    "is_sufficient": False,
    "iteration": 0,
    "max_iterations": 2,    # 設定最多 2 輪迭代（課堂示範用，可調高）
    "final_report": None,
}

print("=" * 60)
print("🚀 啟動 Deep Research Agent")
print(f"   研究問題：{initial_state['research_question']}")
print(f"   最大迭代：{initial_state['max_iterations']} 輪")
print("=" * 60)

# 執行 Graph（串流模式，可以看到每一步的輸出）
final_state = None
for step in graph.stream(initial_state, {"recursion_limit": 25}):
    # 每個 step 是 {node_name: output_dict}
    node_name = list(step.keys())[0]
    print(f"\n{'─' * 40}")
    print(f"  完成節點：{node_name}")
    print(f"{'─' * 40}")
    final_state = step[node_name]

In [ ]:
# ===== 顯示最終研究報告 =====
from IPython.display import Markdown

# 從最後的 graph 執行結果取得 final_report
# 用 invoke 重新執行一次以取得完整 state（或從 stream 結果中取得）
full_result = graph.invoke(initial_state, {"recursion_limit": 25})

print("=" * 60)
print("📄 最終研究報告")
print("=" * 60)
display(Markdown(full_result["final_report"]))

In [ ]:
# ===== 觀察研究過程的統計資訊 =====
print("📊 研究過程統計：")
print(f"   研究問題：{full_result['research_question']}")
print(f"   迭代次數：{full_result['iteration']}")
print(f"   子問題數量：{len(full_result['sub_questions'])}")
print(f"   搜尋結果組數：{len(full_result['search_results'])}")
print(f"   子答案數量：{len(full_result['sub_answers'])}")
print(f"   最終報告長度：{len(full_result['final_report'])} 字")
print(f"   研究是否充分：{full_result['is_sufficient']}")

print(f"\n📋 所有研究子問題：")
for i, q in enumerate(full_result['sub_questions'], 1):
    print(f"   {i}. {q}")

## 2.9 從單一 Agent 邁向 Multi-Agent Swarm

我們剛才實作的是一個**單一 Agent** 的 Deep Research 系統。在實際的複雜研究場景中，可以擴展為 **Multi-Agent** 架構：

### Multi-Agent 架構的優勢

```
                    ┌──────────────┐
                    │  Orchestrator │  (調度者)
                    │   研究主管     │
                    └──────┬───────┘
                           │
            ┌──────────────┼──────────────┐
            │              │              │
    ┌───────▼──────┐ ┌────▼────────┐ ┌───▼──────────┐
    │ 技術研究員    │ │ 產業分析師   │ │ 政策研究員    │
    │ Tech Agent   │ │ Biz Agent   │ │ Policy Agent │
    └──────────────┘ └─────────────┘ └──────────────┘
```

### 如何擴展

1. **專業化分工**：
   - 技術研究員：專注 AI 技術發展、論文分析
   - 產業分析師：專注市場數據、企業案例
   - 政策研究員：專注法規、政府政策
   - 事實查核員：交叉驗證各 Agent 的發現

2. **協作模式**：
   - **順序式**：一個 Agent 完成後，結果交給下一個
   - **並行式**：多個 Agent 同時研究不同面向
   - **辯論式**：Agent 之間互相評審和辯論

### 相關框架

| 框架 | 特色 | 適用場景 |
|------|------|---------|
| **LangGraph** | 圖形化工作流、狀態管理 | 複雜的有狀態 Agent 系統 |
| **CrewAI** | 角色扮演、任務分配 | 團隊協作型 Agent |
| **AutoGen** | 多 Agent 對話 | Agent 之間的討論和辯論 |
| **OpenAI Swarm** | 輕量級 Agent 切換 | 客服、任務路由 |

## 2.10 實作練習

### 練習 1：嘗試不同的研究問題

修改 `research_question`，試試看不同的研究主題：
- "生成式AI對台灣教育體系的影響與機會"
- "全球AI晶片競爭格局與台灣半導體產業的角色"
- "AI Agent 技術的最新發展與商業應用前景"

### 練習 2：調整最大迭代次數

把 `max_iterations` 從 2 調到 3 或 4，觀察：
- 研究品質有什麼變化？
- 每一輪新增了哪些子問題？
- 最終報告的深度有什麼不同？

### 練習 3（進階）：加入事實查核節點

在 `synthesizer_critic` 和 `final_report` 之間加入一個 `fact_checker` 節點：
- 檢查報告中的數據是否有矛盾
- 標記出可能需要進一步驗證的聲明
- 為每個關鍵數據標注可信度

In [ ]:
# ===== 練習 1：嘗試不同的研究問題 =====
# 修改下方的研究問題，重新執行看看！

practice_state = {
    "research_question": "生成式AI對台灣教育體系的影響與機會",  # ← 改成你想研究的問題
    "sub_questions": [],
    "current_sub_question": None,
    "search_results": [],
    "page_contents": [],
    "sub_answers": [],
    "synthesis": None,
    "critique": None,
    "is_sufficient": False,
    "iteration": 0,
    "max_iterations": 2,  # ← 練習 2：試試改成 3 或 4
    "final_report": None,
}

# 取消下方的註解來執行
# practice_result = graph.invoke(practice_state, {"recursion_limit": 25})
# display(Markdown(practice_result["final_report"]))

In [ ]:
# ===== 練習 3（進階）：事實查核節點範例 =====
# 提示：可以這樣定義一個 fact_checker 節點，並加入 graph

def fact_checker_node(state: ResearchState) -> dict:
    """事實查核節點：檢查報告中的數據一致性"""

    synthesis = state.get("synthesis", "")
    sub_answers = state.get("sub_answers", [])

    messages = [
        SystemMessage(content="""你是一位嚴謹的事實查核員。請檢查以下研究報告中的數據和聲明。

請以 JSON 格式回覆：
{
    "verified_claims": ["已驗證的聲明1", "已驗證的聲明2"],
    "questionable_claims": ["需要進一步驗證的聲明1"],
    "contradictions": ["矛盾之處1"],
    "overall_reliability": "高/中/低"
}

只回傳 JSON。"""),
        HumanMessage(content=f"請查核以下研究報告：\n\n{synthesis}")
    ]

    response = llm.invoke(messages)
    print(f"🔎 [Fact Checker] 查核完成")
    print(f"   結果：{response.content[:200]}...")

    # 可以把查核結果加入 critique 中
    return {"critique": state.get("critique", "") + "\n\n事實查核：" + response.content}

# 提示：要使用這個節點，需要重新建立 graph 並加入這個節點
# workflow.add_node("fact_checker", fact_checker_node)
# 然後調整邊的連接方式

print("事實查核節點已定義！可以加入到 graph 中使用。")

---

## 課堂重點回顧

### 今天學到了什麼？

1. **Deep Research 概念**：不同於簡單 Q&A 和 RAG，Deep Research Agent 能主動探索、迭代深化研究
2. **迭代式研究迴圈**：Plan → Search → Read → Synthesize → Evaluate → Loop/Finish
3. **五大核心節點**：
   - Planner：研究規劃、問題拆解
   - Searcher：資料搜尋
   - Reader：內容閱讀與摘要
   - Synthesizer + Critic：彙整與自我評審
   - Final Report：最終報告產出
4. **LangGraph 條件邊**：實現自適應的研究終止邏輯
5. **Multi-Agent 擴展**：從單一 Agent 到多 Agent 協作的演進方向

### 關鍵設計原則

- **TypedDict 管理狀態**：清晰定義 Agent 在每個階段需要的資訊
- **累積式更新**：每個節點只更新自己負責的狀態欄位
- **品質驅動的迭代**：由 Critic 決定是否需要更多研究
- **可擴展架構**：容易加入新節點（如事實查核、翻譯等）